# 05. XGBoost, LightGBM and CatBoost: what each adds

All three implement regularised second-order boosting. Each tree minimises a Taylor approximation
of the loss with gradients $g_i$ and Hessians $h_i$ (for log loss, $g = p - y$, $h = p(1-p)$), plus a
penalty $\gamma|T| + \tfrac12\lambda\sum_j w_j^2$ on leaves. The best leaf weight and the gain of a split are
$$w_j^* = -\frac{G_j}{H_j + \lambda}, \qquad \text{gain} = \tfrac12\Big[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\Big] - \gamma.$$

| | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| tree growth | depth-wise (level by level) | **leaf-wise** (best leaf first) | **symmetric** (same split across a level) |
| split finding | histogram / exact | histogram + GOSS sampling | histogram, oblivious trees |
| categoricals | native (optimal partition) | native (gradient-sorted) | **ordered target statistics** |
| missing values | learned default direction | learned default direction | treated as min/max |

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
from river_health import modeling as M
t = M.prepare(pd.read_parquet(ROOT / "data" / "processed" / "river_features.parquet"))
# a small, readable working set used throughout
F = ["up_stw_dwf_m3d_per_km2", "up_spill_hours_per_km2", "up_lc_urban", "up_lc_arable",
     "up_lc_improved_grass", "up_lc_seminatural", "up_area_km2", "local_n_private_sewage"]
d = t[t.phos_good.notna()].reset_index(drop=True)
X, y = d[F].to_numpy(float), d.phos_good.to_numpy(int)
groups = d.management_catchment.to_numpy()
print(X.shape, "share with good phosphate:", y.mean().round(3))

(3706, 8) share with good phosphate: 0.484


## The XGBoost split gain by hand

In [2]:
p0 = y.mean(); g = p0 - y; h = np.full(len(y), p0 * (1 - p0)); lam = 1.0
j = 0; o = np.argsort(X[:, j]); G, H = np.cumsum(g[o]), np.cumsum(h[o])
GL, HL, GR, HR = G[:-1], H[:-1], G[-1] - G[:-1], H[-1] - H[:-1]
gain = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - G[-1]**2 / (H[-1] + lam))
ok = X[o][1:, j] != X[o][:-1, j]
k = np.argmax(np.where(ok, gain, -np.inf))
print(f"best first split on {F[j]}: <= {(X[o][k, j] + X[o][k + 1, j]) / 2:.3f}, gain {gain[k]:.1f}")
wl, wr = -GL[k] / (HL[k] + lam), -GR[k] / (HR[k] + lam)
print(f"leaf weights (log-odds shifts): left {wl:+.3f}, right {wr:+.3f}")

best first split on up_stw_dwf_m3d_per_km2: <= 4.623, gain 241.6
leaf weights (log-odds shifts): left +0.637, right -0.818


In [3]:
import xgboost as xgb
m = xgb.XGBClassifier(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=1.0, base_score=float(p0),
                      tree_method="exact", min_child_weight=0).fit(X[:, [0]], y)
print(m.get_booster().get_dump(with_stats=True)[0])

0:[f0<4.62276936] yes=1,no=2,missing=1,gain=483.22937,cover=925.592285
	1:leaf=0.636584103,cover=520.739319
	2:leaf=-0.81835258,cover=404.852966



The library finds the same threshold and the same leaf weights (`leaf=` values) as the formula. Its
reported `gain` is exactly twice ours: XGBoost's dump omits the ½. `missing=1` is the learned default
direction for missing values.

## Leaf-wise vs depth-wise growth

With the same number of leaves, LightGBM spends them where the gain is largest, so its trees can be deep and
lopsided. That makes it fast and accurate, but more prone to overfit small data unless `num_leaves` and
`min_child_samples` are limited. Below, both use a tiny minimum leaf size so the difference shows.

In [4]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
folds = list(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X, y, groups))
def cv(model):
    o = np.zeros(len(y))
    for a, b in folds: o[b] = model.fit(X[a], y[a]).predict_proba(X[b])[:, 1]
    return roc_auc_score(y, o)
rows = []
for leaves in [4, 8, 16, 32, 64, 128]:
    rows.append(("LightGBM leaf-wise", leaves, cv(lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=leaves,
                                                                    min_child_samples=5, verbose=-1, random_state=0))))
    depth = int(np.log2(leaves))
    rows.append(("XGBoost depth-wise", leaves, cv(xgb.XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=depth,
                                                                   min_child_weight=1, random_state=0))))
r = pd.DataFrame(rows, columns=["growth", "max leaves", "AUC"])
r.pivot(index="max leaves", columns="growth", values="AUC").round(3)

growth,LightGBM leaf-wise,XGBoost depth-wise
max leaves,,
4,0.804,0.801
8,0.806,0.804
16,0.804,0.807
32,0.802,0.805
64,0.802,0.804
128,0.800,0.803


On 3,700 rows the two growth strategies score within 0.01 of each other. LightGBM peaks with about 8 leaves
and declines slowly as leaves are added, which is the mild overfitting expected from leaf-wise growth.
Shrinkage and 300 small steps keep both in check.

## Categorical features: ordered target statistics (CatBoost)

Replacing a category with the mean target of its rows ("target encoding") leaks the row's own label into its
feature. CatBoost encodes each row using only rows *before it* in a random permutation, the same trick it
uses to compute gradients ("ordered boosting"). Below, management catchment (90 levels) is target-encoded
naively and in the ordered way.

In [5]:
cat = d.management_catchment.to_numpy()
def naive_te(cat, y):
    s = pd.Series(y).groupby(cat).transform("mean").to_numpy(); return s
def ordered_te(cat, y, prior=y.mean(), a=1.0, seed=0):
    rng = np.random.default_rng(seed); o = rng.permutation(len(y)); out = np.zeros(len(y))
    sums, cnts = {}, {}
    for i in o:
        c = cat[i]; out[i] = (sums.get(c, 0) + a * prior) / (cnts.get(c, 0) + a)
        sums[c] = sums.get(c, 0) + y[i]; cnts[c] = cnts.get(c, 0) + 1
    return out
for name, enc in [("naive", naive_te(cat, y)), ("ordered", ordered_te(cat, y))]:
    print(f"{name:8s} in-sample AUC of the encoding alone: {roc_auc_score(y, enc):.3f}")

naive    in-sample AUC of the encoding alone: 0.800
ordered  in-sample AUC of the encoding alone: 0.758


The naive encoding scores 0.80 in-sample against 0.76 for the ordered one, because each row partly sees its own label. A tree built on it
would learn that inflated signal and disappoint on new data. (This project deliberately keeps management
catchment *out* of the features, because it defines the spatial CV groups.)

## Missing values

Trees in XGBoost and LightGBM learn a default direction for missing values at each split. Measured chemistry
is missing for 27% of rivers, and those rivers are *more* often good (0.60 vs 0.44): monitoring targets
problem rivers. The model can use missingness itself as information, which helps prediction but is a
reminder that monitoring is not random.

In [6]:
chem = d.chem_orthophosphate_mgP.to_numpy()
Xc = np.c_[X, chem]
print(f"rivers without phosphate monitoring: {np.isnan(chem).mean():.0%}; "
      f"P(good) with data {y[~np.isnan(chem)].mean():.2f}, without {y[np.isnan(chem)].mean():.2f}")
for lab, XX in [("pressures", X), ("+ measured phosphate (with NaN)", Xc)]:
    o = np.zeros(len(y))
    for a, b in folds: o[b] = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, verbose=-1).fit(XX[a], y[a]).predict_proba(XX[b])[:, 1]
    print(f"{lab:35s} AUC {roc_auc_score(y, o):.3f}")

rivers without phosphate monitoring: 27%; P(good) with data 0.44, without 0.60


pressures                           AUC 0.808


+ measured phosphate (with NaN)     AUC 0.894
